In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers \
    faiss-cpu pypdf pdfplumber gradio rouge-score nltk bert-score \
    langchain langchain-text-splitters \
    rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.3 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import time, re, json, math, warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
from pypdf import PdfReader
from bert_score import score as bert_score_fn
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
import faiss
import nltk
from rouge_score import rouge_scorer as rs_module
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All libraries imported.")
print(f"CUDA available: {torch.cuda.is_available()}")

All libraries imported.
CUDA available: True


In [3]:
MODEL_LABEL       = "Qwen2.5-1.5B-Instruct (Local 4-bit)"
MODEL_DISPLAY_NAME = "Qwen2.5-1.5B-Instruct"
MODEL_LOG         = "qwen_evaluation_log"
MODEL_NAME        = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model with 4-bit NF4 quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)
model.eval()
print(f"Model loaded: {MODEL_NAME}")

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model with 4-bit NF4 quantization...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct


In [4]:
# v6: Upgraded to BGE-Large embedding + BGE Reranker v2 m3
EMBED_MODEL  = "BAAI/bge-large-en-v1.5"     # 335M params, 1024-dim — stronger than MiniLM
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"    # State-of-art cross-encoder reranker

print(f"Loading bi-encoder   : {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Loading cross-encoder: {RERANK_MODEL}")
reranker = CrossEncoder(RERANK_MODEL)
print("Retrieval models ready.")
print(f"  Embedding dim : {embedder.get_sentence_embedding_dimension()}")

Loading bi-encoder   : BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Retrieval models ready.
  Embedding dim : 1024


In [5]:
from rank_bm25 import BM25Okapi
import pdfplumber

# ── Section/topic metadata detection ────────────────────────────────────────
_META_PATTERNS = [
    (r"(?i)\bsection\s+(\d+[A-Za-z]?)\b",             "Section {}"),
    (r"(?i)\bpart\s+(I{{1,3}}V?|VI{{0,3}}|\d+)\b",    "Part {}"),
    (r"(?i)\b(medical benefit|sickness|treatment|injur)", "Medical Benefits"),
    (r"(?i)\b(death benefit|burial|deceased)",            "Death & Burial Benefits"),
    (r"(?i)\b(disabilit)",                                "Disability Benefits"),
    (r"(?i)\b(repatri)",                                   "Repatriation"),
    (r"(?i)\b(overtime|working hours)",                   "Working Hours & Overtime"),
    (r"(?i)\b(basic salary|basic wage|monthly salary)",   "Wages & Salary"),
    (r"(?i)\b(allotment|remittance)",                     "Allotment & Remittance"),
    (r"(?i)\b(terminat|dismissal|disciplin)",             "Termination & Discipline"),
    (r"(?i)\b(insurance|coverage|personal accident)",     "Insurance"),
    (r"(?i)\b(placement fee|recruitment fee)",            "Placement & Recruitment"),
    (r"(?i)\b(due process|right to be heard)",            "Due Process"),
    (r"(?i)\b(beneficiar|qualified dependent)",           "Beneficiaries"),
    (r"(?i)\b(obligation|employer shall|ship owner)",     "Employer Obligations"),
]

def _detect_section(text: str) -> str:
    probe = text[:200]
    for pattern, label in _META_PATTERNS:
        m = re.search(pattern, probe)
        if m:
            groups = m.groups()
            return label.format(groups[0].strip().title()) if "{}" in label and groups else label
    return "General"

# ── Layout-aware PDF extractor (column-aware) ────────────────────────────────
def load_and_clean_pdf(pdf_path: str) -> str:
    full_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            w, h = page.width, page.height
            for bbox in [(0, 0, w / 2, h), (w / 2, 0, w, h)]:
                col = page.crop(bbox)
                raw = col.extract_text()
                if raw:
                    raw = raw.replace("-\n", "").replace("\n", " ")
                    raw = re.sub(r"Page \d+ of \d+", "", raw)
                    raw = re.sub(r"[ \t]+", " ", raw).strip()
                    if raw:
                        full_text.append(raw)
    return "\n\n".join(full_text)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

PDF_PATH    = "/content/seaman.pdf"
pdf_text    = load_and_clean_pdf(PDF_PATH)
raw_chunks  = [c.strip() for c in splitter.split_text(pdf_text) if len(c.strip()) > 80]

# ── Attach metadata to every chunk ──────────────────────────────────────────
chunk_texts = []
chunk_meta  = []       # parallel list — same index as chunk_texts
for c in raw_chunks:
    chunk_texts.append(c)
    chunk_meta.append({"section": _detect_section(c)})

print(f"Total characters : {len(pdf_text):,}")
print(f"Total chunks     : {len(chunk_texts)}")
print(f"Avg chunk length : {sum(len(c) for c in chunk_texts)//len(chunk_texts)} chars")
sections_found = set(m["section"] for m in chunk_meta)
print(f"Unique sections  : {len(sections_found)} — {sorted(sections_found)}")

# ── Dense index (FAISS) ──────────────────────────────────────────────────────
print("Generating BGE-Large dense embeddings...")
chunk_embeddings = embedder.encode(
    chunk_texts, show_progress_bar=True, batch_size=32, normalize_embeddings=True
)
dim   = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings.astype(np.float32))
print(f"FAISS index  : {index.ntotal} vectors  ({dim}-dim)")

# ── Sparse index (BM25) ──────────────────────────────────────────────────────
print("Building BM25 sparse index...")
tokenized_corpus = [c.lower().split() for c in chunk_texts]
bm25_index       = BM25Okapi(tokenized_corpus)
print(f"BM25 index   : {len(tokenized_corpus)} documents")
print("All indexes ready — v6.")

Total characters : 547,722
Total chunks     : 997
Avg chunk length : 628 chars
Unique sections  : 2 — ['General', 'Medical Benefits']
Generating BGE-Large dense embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

FAISS index  : 997 vectors  (1024-dim)
Building BM25 sparse index...
BM25 index   : 997 documents
All indexes ready — v6.


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# LAYERED ANTI-HALLUCINATION DEFENSE — SeamanBot v6
# ───────────────────────────────────────────────────────────────────────────────
#  Layer 1   BGE-Large bi-encoder   BAAI/bge-large-en-v1.5 (1024-dim)
#  Layer 1b  Hybrid retrieval       BM25 + FAISS → RRF fusion, k_init=80
#  Layer 1c  Multi-query expansion  3 template-based query reformulations (Qwen)
#  Layer 1d  Stem keyword boost     6-char prefix matching
#  Layer 1e  BGE Reranker v2-m3     reranks merged pool, k_final=5
#  Layer 1f  MMR diversity          lambda=0.70
#  Layer 2   Metadata-aware prompt  chunk section labels injected into context
#  Layer 3   Prompt constraints     forbidden refusal phrases, STRICT fallback
#  Layer 4   Decoding params        rep_penalty=1.15, no_repeat_ngram=4 [Qwen]
#  Layer 5   Retry gate             retry only if faith improves >= 0.05
# ═══════════════════════════════════════════════════════════════════════════════

_STOPWORDS = {
    "what","is","the","a","an","of","for","to","in","on","are","be","by","at",
    "as","if","or","and","how","who","does","do","can","was","were","will","with",
    "from","that","this","it","its","they","their","when","under","before","after",
    "must","shall","should","would","any","all","been","have","has","not","no",
    "which","about","into","also","upon","may","per","given","such","during",
    "give","get","set","let","put","take","make","use","than","then","each",
    "some","more","very","just","only","here","there","where","case","cases",
}

def _cos(a, b):
    a, b = np.array(a, dtype=np.float32), np.array(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

MARITIME_SYNONYMS = {
    r"\bcontract\b"                           : "contract employment term duration period agreement",
    r"\bsalary\b|\bwage\b"                 : "salary wage pay basic compensation monthly rate",
    r"\bmedical\b"                            : "medical sickness injury treatment illness hospitalization medicine",
    r"\brepatri\w+"                           : "repatriation repatriated return passage homeward cost expense transport",
    r"\bdeath\b|\bdi(ed|es)\b"             : "death deceased die died burial compensation beneficiary",
    r"\bdisciplin\w+"                         : "disciplinary discipline offense misconduct penalty dismiss sanction violation",
    r"\bovertime\b"                           : "overtime hours work additional pay rate beyond",
    r"\binsurance\b"                          : "insurance coverage protection life accident personal",
    r"\bterminat\w+"                          : "termination pre-termination dismissal grounds cause end",
    r"\bdisability\b|\bdisabled\b"         : "disability disabled permanent total partial grade schedule injury",
    r"\ballotment\b"                          : "allotment remittance family beneficiary percent percentage monthly mandatory",
    r"\bleave\b"                              : "leave vacation rest days paid entitlement annual",
    r"\bplacement\b"                          : "placement fee recruitment agency prohibited illegal banned",
    r"\bbenefit\w*"                           : "benefit benefits compensation payment entitlement",
    r"\bseafarer\b"                           : "seafarer crew member mariner employee worker",
    r"\bemployer\b"                           : "employer owner company agency principal obligation shall",
    r"\bdocument\w*"                          : "documents document copy given provided contract departure embarkation",
    r"\bdeparture\b"                          : "departure embarkation sign-off prior departure leaving document copy",
    r"\bworking\s+hours\b|\bwork\s+hours\b|\bhours\b": "working hours eight regular daily standard per day",
    r"\bdue\s+process\b"                    : "due process dismiss hearing explain opportunity informed",
    r"\b120\b"                               : "120 days hundred treatment disability assessment period beyond",
    r"\ballowanc\w+"                         : "allowance subsistence vacation leave pay entitlement additional",
    r"\breimburs\w+"                         : "reimbursement reimburse expense cost shoulder pay",
    r"\bliable\b|\bliability\b"            : "liable liability responsible obligation penalty",
    r"\bbeneficiar\w+"                       : "beneficiary beneficiaries qualified spouse children dependent family",
    r"\bsickness\b|\bill\w*\b"            : "sickness sick illness injury allowance treatment basic wage rate",
    r"\bfee\w*"                              : "fee fees placement recruitment prohibited banned",
    r"\bobligat\w+"                          : "obligation obligations employer shall must provide duty responsible",
    r"\bpercent\b|\b%\b"                   : "percent percentage allotment mandatory minimum salary portion",
    r"\bfail\b|\bfails\b|\bfailed\b"     : "fail fails failed failure employer repatriation liable responsible",
    r"\bshoulder\w*"                         : "shoulder cost expense pay responsible employer repatriation",
    r"\bqualified\b"                         : "qualified beneficiary spouse children dependent entitled",
    r"\bstandard\b"                          : "standard regular normal working hours eight daily",
    r"\bcondit\w+"                           : "conditions terms when may circumstances case grounds",
    r"\bground\w*"                           : "grounds conditions cause reasons justification termination",
    r"\bobligation\w*"                       : "obligation duty shall provide employer required",
}

def expand_query(query: str) -> str:
    expanded = query
    for pattern, expansion in MARITIME_SYNONYMS.items():
        if re.search(pattern, query, re.IGNORECASE):
            expanded += " " + expansion
    return expanded


# ── Multi-Query: 3 template reformulations (Qwen — no extra LLM call needed) ─
def _multi_query(query: str) -> list:
    stems = [w for w in re.findall(r"\b[a-zA-Z]{4,}\b", query)
             if w.lower() not in _STOPWORDS]
    kw = " ".join(stems[:6])
    return [
        query,
        f"What does the POEA Standard Employment Contract state regarding {kw}?",
        f"Under the POEA SEC for Filipino seafarers, what are the provisions on {kw}?",
        expand_query(query),
    ]


# ── Hybrid retrieval: BM25 + FAISS → RRF ────────────────────────────────────
def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank)

def retrieve_context(query: str, k_init: int = 80, k_final: int = 5) -> tuple:
    queries   = _multi_query(query)            # Layer 1c: multi-query expansion
    expanded  = expand_query(query)

    # ── FAISS: embed all query variants, union results ──
    q_embs    = embedder.encode(queries, normalize_embeddings=True).astype(np.float32)
    faiss_seen, faiss_ranks_union = set(), {}
    for qi, qe in enumerate(q_embs):
        _, I = index.search(qe.reshape(1, -1), min(k_init, len(chunk_texts)))
        for r, idx in enumerate(I[0]):
            if 0 <= idx < len(chunk_texts):
                c = chunk_texts[idx]
                if c not in faiss_seen:
                    faiss_ranks_union[c] = r + 1
                    faiss_seen.add(c)
    main_q_emb = q_embs[0]

    # ── BM25 sparse retrieval ──
    bm25_scores  = bm25_index.get_scores(expanded.lower().split())
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:k_init]
    bm25_ranks   = {chunk_texts[i]: r + 1
                    for r, i in enumerate(bm25_top_idx) if 0 <= i < len(chunk_texts)}

    # ── RRF fusion ──
    all_chunks = set(faiss_ranks_union) | set(bm25_ranks)
    rrf = {c: (_rrf_score(faiss_ranks_union.get(c, k_init + 1)) +
               _rrf_score(bm25_ranks.get(c, k_init + 1)))
           for c in all_chunks}
    pool = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k_init]

    # ── Stem keyword boost ──
    query_stems = [w[:6].lower() for w in re.findall(r"\b[a-zA-Z]{4,}\b", query)
                   if w.lower() not in _STOPWORDS]
    seen = set(id(c) for c in pool)
    for chunk in chunk_texts:
        if id(chunk) not in seen and any(s in chunk.lower() for s in query_stems):
            pool.append(chunk)
            seen.add(id(chunk))

    # ── BGE Reranker v2-m3 ──
    if len(pool) > 1:
        rerank_pool = pool[:120]
        ce_scores   = reranker.predict([[query, c] for c in rerank_pool])
        ranked      = sorted(zip(ce_scores, rerank_pool), reverse=True)
        top_sim     = float(1.0 / (1.0 + math.exp(-ranked[0][0])))
        ce_pool     = [c for _, c in ranked[:k_final * 3]]
    else:
        ce_pool, top_sim = pool[:k_final], 0.5

    # ── MMR diversity (lambda=0.70) ──
    if len(ce_pool) > k_final:
        c_embs   = embedder.encode(ce_pool, normalize_embeddings=True)
        selected, remaining = [], list(range(len(ce_pool)))
        for _ in range(k_final):
            if not remaining: break
            if not selected:
                scores = [_cos(main_q_emb, c_embs[i]) for i in remaining]
            else:
                scores = [0.70 * _cos(main_q_emb, c_embs[i])
                          - 0.30 * max(_cos(c_embs[i], c_embs[j]) for j in selected)
                          for i in remaining]
            best = remaining[int(np.argmax(scores))]
            selected.append(best)
            remaining.remove(best)
        final_chunks = [ce_pool[i] for i in selected]
    else:
        final_chunks = ce_pool[:k_final]

    # ── Build metadata-tagged context string (Layer 2) ──
    parts = []
    for ch in final_chunks:
        try:
            idx = chunk_texts.index(ch)
            sec = chunk_meta[idx]["section"]
        except ValueError:
            sec = "General"
        parts.append(f"[Source: {sec}]\n{ch}")
    context_str = "\n\n---\n\n".join(parts)
    return context_str, final_chunks, top_sim


# ── Prompts ───────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an expert Philippine maritime labor law assistant specializing in "
    "the POEA Standard Employment Contract (SEC) for Filipino seafarers.\n"
    "Each context block begins with [Source: <section>] indicating which part of the "
    "contract it comes from. Use this metadata to cite the correct section in your answer.\n"
    "RULES:\n"
    "1. Answer ONLY using information explicitly stated in the provided context.\n"
    "2. NEVER say: 'I cannot find', 'not mentioned', 'context does not contain', "
    "   'not in context', 'not provided'. These phrases are forbidden.\n"
    "3. Cite the source section (e.g. 'Under Medical Benefits...') when visible.\n"
    "4. If the context gives partial information, use it to construct the best possible answer."
)

STRICT_PROMPT = (
    "You are a strict POEA SEC legal assistant. "
    "Use ONLY sentences directly verifiable in the provided context blocks. "
    "Each block starts with [Source: section]. Cite the section in your answer. "
    "No external knowledge. No speculation."
)

def _faith_inline(answer: str, chunks: list) -> float:
    if not answer.strip() or not chunks: return 0.0
    sents  = [s.strip() for s in re.split(r"[.!?\n]", answer) if len(s.strip()) > 12]
    c_embs = embedder.encode(chunks, normalize_embeddings=True)
    if not sents:
        a_emb = embedder.encode([answer], normalize_embeddings=True)[0]
        return float(max(_cos(a_emb, c) for c in c_embs))
    return float(np.mean([
        max(_cos(embedder.encode([s], normalize_embeddings=True)[0], c) for c in c_embs)
        for s in sents
    ]))


def generate_answer(query: str) -> dict:
    if not query or not query.strip():
        return {"answer": "Please enter a valid question.", "context": "",
                "chunks": [], "latency_sec": 0.0, "confidence": 0.0}
    t_start = time.time()
    try:
        context, chunks, confidence = retrieve_context(query)
    except Exception as e:
        return {"answer": f"Retrieval error: {e}", "context": "", "chunks": [],
                "latency_sec": 0.0, "confidence": 0.0}

    def _call(q_text: str, sys_p: str) -> str:
        msgs   = [{"role": "system", "content": sys_p},
                  {"role": "user", "content": (
                      f"Context from official POEA documents:\n---\n{context}\n---\n\n"
                      f"Question: {q_text}\n\nAnswer:")}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=3072).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.2,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.15,
                no_repeat_ngram_size=4,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()

    try:
        answer = _call(query, SYSTEM_PROMPT)
    except Exception as e:
        return {"answer": f"Generation error: {e}", "context": context,
                "chunks": chunks, "latency_sec": 0.0, "confidence": confidence}

    # Layer 5: retry gate — only if faithfulness improves by >= 0.05
    faith_orig = _faith_inline(answer, chunks)
    if faith_orig < 0.35:
        try:
            alt       = _call(query, STRICT_PROMPT)
            faith_alt = _faith_inline(alt, chunks)
            if faith_alt >= faith_orig + 0.05:
                answer = alt
        except Exception:
            pass

    latency = round(time.time() - t_start, 2)
    return {"answer": answer, "context": context, "chunks": chunks,
            "latency_sec": latency, "confidence": confidence}

print(f"Pipeline ready — {MODEL_LABEL}")

Pipeline ready — Qwen2.5-1.5B-Instruct (Local 4-bit)


In [7]:
BERTSCORE_MODEL = "distilbert-base-uncased"
_bs_cache = {}

def _cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def compute_recall_at_k(retrieved_chunks: list, keywords: list) -> float:
    if not keywords: return 1.0
    combined = " ".join(retrieved_chunks).lower()
    found = sum(1 for kw in keywords if kw.lower() in combined)
    return round(found / len(keywords), 4)

def compute_precision_at_k(retrieved_chunks: list, keywords: list) -> float:
    if not retrieved_chunks or not keywords: return 0.0
    relevant = sum(
        1 for chunk in retrieved_chunks
        if any(kw.lower() in chunk.lower() for kw in keywords)
    )
    return round(relevant / len(retrieved_chunks), 4)

def compute_context_relevancy(query: str, chunks: list) -> float:
    if not chunks: return 0.0
    q_emb  = embedder.encode([query], normalize_embeddings=True)[0]
    c_embs = embedder.encode(chunks, normalize_embeddings=True)
    return round(float(np.mean([_cosine(q_emb, c) for c in c_embs])), 4)

def compute_faithfulness(answer: str, chunks: list) -> float:
    if not answer.strip() or not chunks: return 0.0
    sentences = [s.strip() for s in re.split(r'[.!?\n]', answer) if len(s.strip()) > 12]
    c_embs = embedder.encode(chunks, normalize_embeddings=True)
    if not sentences:
        a_emb = embedder.encode([answer], normalize_embeddings=True)[0]
        return round(float(max(_cosine(a_emb, c) for c in c_embs)), 4)
    scores = []
    for sent in sentences:
        s_emb = embedder.encode([sent], normalize_embeddings=True)[0]
        scores.append(max(_cosine(s_emb, c) for c in c_embs))
    return round(float(np.mean(scores)), 4)

def compute_answer_relevancy(query: str, answer: str) -> float:
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    a_emb = embedder.encode([answer], normalize_embeddings=True)[0]
    return round(_cosine(q_emb, a_emb), 4)

def compute_rouge_l(hypothesis: str, reference: str) -> float:
    if not hypothesis.strip() or not reference.strip(): return 0.0
    scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)
    return round(scorer.score(reference, hypothesis)['rougeL'].fmeasure, 4)

def compute_bert_score(answer: str, reference: str) -> float:
    if not answer.strip() or not reference.strip(): return 0.0
    cache_key = (answer[:200], reference[:200])
    if cache_key in _bs_cache:
        return _bs_cache[cache_key]
    try:
        _, _, F1 = bert_score_fn(
            [answer], [reference],
            model_type=BERTSCORE_MODEL, lang="en", verbose=False
        )
        val = round(float(F1[0]), 4)
    except Exception:
        val = 0.0
    _bs_cache[cache_key] = val
    return val

print("Evaluation metric functions ready.")
print("Metrics: Recall@K | Precision@K | CtxRel | Faithfulness | AnsRel | ROUGE-L | BERTScore | Latency")


Evaluation metric functions ready.
Metrics: Recall@K | Precision@K | CtxRel | Faithfulness | AnsRel | ROUGE-L | BERTScore | Latency


In [8]:
TEST_SET = [
    {"query": "What is the maximum duration of a seafarer's employment contract?",
     "keywords": ["month", "contract", "duration"],
     "reference_answer": "The employment contract of a seafarer shall be for a period not exceeding twelve months. The exact duration is agreed upon and stated in the contract."},
    {"query": "What are the conditions for renewing or extending a seafarer's employment contract?",
     "keywords": ["extend", "renew", "mutual", "agreement"],
     "reference_answer": "A seafarer's contract may be extended by mutual agreement of both parties. Any extension must be in accordance with POEA rules and must not exceed the allowable period."},
    {"query": "What documents must be given to a seafarer before departure?",
     "keywords": ["contract", "departure", "document", "copy"],
     "reference_answer": "Before departure, the seafarer must be provided a copy of the employment contract and all relevant documents covering the terms and conditions of employment."},
    {"query": "How is the basic monthly salary of a seafarer determined?",
     "keywords": ["basic", "salary", "wage", "monthly"],
     "reference_answer": "The basic monthly salary is fixed in the employment contract based on the seafarer's position and rank, and must not be lower than the minimum prescribed by the POEA."},
    {"query": "Is a seafarer entitled to overtime pay?",
     "keywords": ["overtime", "hours", "pay"],
     "reference_answer": "Yes. A seafarer is entitled to overtime pay for work performed beyond the regular eight hours a day. Overtime is compensated at a rate fixed in the contract."},
    {"query": "What allowances or additional pay is a seafarer entitled to receive?",
     "keywords": ["allowance", "subsistence", "leave", "pay"],
     "reference_answer": "Seafarers are entitled to various allowances including subsistence allowance, vacation leave pay, and other benefits as specified in the POEA Standard Employment Contract."},
    {"query": "How and when must the seafarer's wages be paid?",
     "keywords": ["wages", "paid", "allotment", "remit"],
     "reference_answer": "The seafarer's wages must be paid regularly. A portion of the wage, the allotment, is remitted to the seafarer's designated beneficiary in the Philippines."},
    {"query": "What medical benefits are available to seafarers who get injured on board?",
     "keywords": ["medical", "treatment", "sickness", "injury"],
     "reference_answer": "A seafarer who is injured on board is entitled to necessary medical treatment at the employer's expense, including hospitalization, surgery, and medication."},
    {"query": "What is the sickness allowance rate for an injured or ill seafarer?",
     "keywords": ["sickness", "allowance", "basic", "wage"],
     "reference_answer": "The sickness allowance is equivalent to the seafarer's basic wage during medical treatment. It is paid for the period specified in the contract."},
    {"query": "What happens if a seafarer's illness or injury requires treatment beyond 120 days?",
     "keywords": ["120", "days", "disability", "treatment"],
     "reference_answer": "If treatment extends beyond 120 days, the seafarer may be assessed for disability benefits and compensation under the POEA schedule of disability allowances."},
    {"query": "What disability benefits is a seafarer entitled to if permanently injured?",
     "keywords": ["disability", "permanent", "compensation", "grade"],
     "reference_answer": "A permanently injured seafarer is entitled to disability compensation based on the POEA schedule of disability allowances. The maximum amount is for total permanent disability."},
    {"query": "What death benefits are provided if a seafarer dies during the contract?",
     "keywords": ["death", "benefit", "burial", "compensation"],
     "reference_answer": "If a seafarer dies during the contract, the employer shall pay death compensation to the beneficiaries. Burial assistance is also provided as specified in the contract."},
    {"query": "Who are the qualified beneficiaries in case of a seafarer's death?",
     "keywords": ["beneficiar", "spouse", "children", "dependent"],
     "reference_answer": "The qualified beneficiaries are the seafarer's legal spouse, legitimate or illegitimate children, and in their absence, the seafarer's parents or other dependents."},
    {"query": "What insurance coverage is provided to Filipino seafarers?",
     "keywords": ["insurance", "accident", "coverage", "life"],
     "reference_answer": "The employer must provide the seafarer with compulsory accident and life insurance coverage for the duration of the contract as required by POEA regulations."},
    {"query": "Is the employer required to provide personal accident insurance to the seafarer?",
     "keywords": ["insurance", "employer", "accident", "required"],
     "reference_answer": "Yes. The employer is required to provide the seafarer with personal accident insurance covering death, permanent disability, and related contingencies."},
    {"query": "Under what conditions can a seafarer be repatriated?",
     "keywords": ["repatri", "expir", "medical", "termination"],
     "reference_answer": "A seafarer may be repatriated upon expiration of the contract, for medical reasons, upon termination of employment, or when the vessel is sold or becomes unseaworthy."},
    {"query": "Who shoulders the cost of repatriation for a seafarer?",
     "keywords": ["cost", "repatri", "employer", "expense"],
     "reference_answer": "The cost of repatriation is shouldered by the employer. The seafarer shall not be charged for repatriation expenses unless responsible for their own repatriation."},
    {"query": "What happens if the employer fails to repatriate the seafarer?",
     "keywords": ["fail", "repatri", "employer", "liable"],
     "reference_answer": "If the employer fails to repatriate the seafarer, the employer shall be held liable for all costs incurred and subject to sanctions under POEA rules and regulations."},
    {"query": "What are the grounds for termination of a seafarer's contract before completion?",
     "keywords": ["terminat", "dismissal", "cause", "ground"],
     "reference_answer": "Pre-termination may be based on just causes such as serious misconduct, willful disobedience, habitual neglect of duty, or commission of a crime while on board."},
    {"query": "What disciplinary actions can be taken against a seafarer for misconduct?",
     "keywords": ["disciplin", "misconduct", "offense", "penalty"],
     "reference_answer": "Disciplinary actions include written warnings, fines, suspension, and dismissal depending on the gravity of the offense. The seafarer must be given the right to due process."},
    {"query": "What due process rights does a seafarer have before being dismissed?",
     "keywords": ["due process", "dismiss", "hear", "explain"],
     "reference_answer": "Before dismissal, the seafarer has the right to be informed of the charge and given an opportunity to explain or defend themselves before any action takes effect."},
    {"query": "What are the obligations of the ship owner or employer toward the seafarer?",
     "keywords": ["employer", "shall", "provide", "obligation"],
     "reference_answer": "The employer is obligated to provide safe working conditions, timely payment of wages, medical care, adequate food and accommodation, and repatriation upon contract completion."},
    {"query": "What are the standard working hours for seafarers under the POEA contract?",
     "keywords": ["hours", "work", "eight", "regular"],
     "reference_answer": "The standard working hours are eight hours per day. Work beyond eight hours is considered overtime and must be compensated at the overtime rate specified in the contract."},
    {"query": "Is a seafarer allowed to pay placement or recruitment fees?",
     "keywords": ["placement", "fee", "prohibit", "recruit"],
     "reference_answer": "No. It is prohibited for a seafarer to pay placement fees. The costs of recruitment and placement must be shouldered by the employer or recruitment agency, not the seafarer."},
    {"query": "What is the mandatory allotment for a seafarer's monthly salary?",
     "keywords": ["allotment", "percent", "salary", "family"],
     "reference_answer": "The seafarer is required to allot a portion of their basic monthly salary to their designated beneficiary. The minimum allotment is set by POEA regulations."},
]
print(f"Test set loaded: {len(TEST_SET)} queries.")


Test set loaded: 25 queries.


In [9]:
evaluation_results = []
print(f"Running evaluation — {MODEL_LABEL}\n")
print("=" * 100)

for i, tc in enumerate(TEST_SET):
    q, ref, kws = tc["query"], tc["reference_answer"], tc["keywords"]
    print(f"\n[Query {i+1}/{len(TEST_SET)}] {q}")

    result  = generate_answer(q)
    answer  = result["answer"]
    chunks  = result["chunks"]
    latency = result["latency_sec"]

    row = {
        "query": q, "answer": answer, "reference_answer": ref,
        "recall_at_k"      : compute_recall_at_k(chunks, kws),
        "precision_at_k"   : compute_precision_at_k(chunks, kws),
        "context_relevancy": compute_context_relevancy(q, chunks),
        "faithfulness"     : compute_faithfulness(answer, chunks),
        "answer_relevancy" : compute_answer_relevancy(q, answer),
        "rouge_l"          : compute_rouge_l(answer, ref),
        "bert_score"       : compute_bert_score(answer, ref),
        "latency_sec"      : latency,
    }
    evaluation_results.append(row)
    disp = (answer[:110] + "...") if len(answer) > 110 else answer
    print(f"  Answer   : {disp}")
    print(f"  Rec@K: {row['recall_at_k']:.4f} | Prec@K: {row['precision_at_k']:.4f} | "
          f"CtxRel: {row['context_relevancy']:.4f} | Faith: {row['faithfulness']:.4f} | "
          f"AnsRel: {row['answer_relevancy']:.4f} | ROUGE-L: {row['rouge_l']:.4f} | "
          f"BERTScore: {row['bert_score']:.4f} | Lat: {latency}s")

print("\n" + "=" * 100)
print("Evaluation complete.")


Running evaluation — Qwen2.5-1.5B-Instruct (Local 4-bit)


[Query 1/25] What is the maximum duration of a seafarer's employment contract?


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Answer   : No, according to the given context, a seafearer is not allowed to pay any placement or recruitment fee. The re...
  Rec@K: 0.0000 | Prec@K: 0.0000 | CtxRel: 0.5546 | Faith: 0.6276 | AnsRel: 0.8648 | ROUGE-L: 0.1649 | BERTScore: 0.7858 | Lat: 15.1s

[Query 25/25] What is the mandatory allotment for a seafarer's monthly salary?


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Answer   : The Mandatory Allotment for a Seafarer's Monthly Salary under the POEA standard employment contract (SEC) shal...
  Rec@K: 0.0000 | Prec@K: 0.0000 | CtxRel: 0.5446 | Faith: 0.5367 | AnsRel: 0.8226 | ROUGE-L: 0.0915 | BERTScore: 0.7532 | Lat: 20.9s

Evaluation complete.


In [10]:
metrics_keys = ["recall_at_k", "precision_at_k", "context_relevancy",
                "faithfulness", "answer_relevancy", "rouge_l", "bert_score", "latency_sec"]

print(f"\n{'=' * 62}")
print(f"AGGREGATE RESULTS — {MODEL_LABEL}")
print(f"{'=' * 62}")
for key in metrics_keys:
    val  = np.mean([r[key] for r in evaluation_results])
    flag = " ✓" if val >= 0.60 else (" ~" if val >= 0.40 else " ✗")
    print(f"  {key:<25}: {val:.4f}{flag}")
print(f"{'=' * 62}")

print("\nDETAILED PER-QUERY TABLE")
print("-" * 148)
print(f"{'#':<4} {'Query (truncated)':<45} {'Rec@K':<8} {'Prec@K':<8} "
      f"{'CtxRel':<8} {'Faith':<8} {'AnsRel':<8} {'ROUGE-L':<9} {'BERTSc':<9} Lat(s)")
print("-" * 148)
for i, r in enumerate(evaluation_results):
    q = (r['query'][:43] + "..") if len(r['query']) > 45 else r['query']
    print(f"{i+1:<4} {q:<45} "
          f"{r['recall_at_k']:<8.4f} {r['precision_at_k']:<8.4f} "
          f"{r['context_relevancy']:<8.4f} {r['faithfulness']:<8.4f} "
          f"{r['answer_relevancy']:<8.4f} {r['rouge_l']:<9.4f} "
          f"{r['bert_score']:<9.4f} {r['latency_sec']:.2f}")
print("-" * 148)
print(f"{'MEAN':<50}", end="")
for key in metrics_keys:
    print(f"{np.mean([r[key] for r in evaluation_results]):<9.4f} ", end="")
print()

with open(f"/content/{MODEL_LOG}.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)
print(f"\nSaved: /content/{MODEL_LOG}.json")


AGGREGATE RESULTS — Qwen2.5-1.5B-Instruct (Local 4-bit)
  recall_at_k              : 0.1067 ✗
  precision_at_k           : 0.0960 ✗
  context_relevancy        : 0.5646 ~
  faithfulness             : 0.5502 ~
  answer_relevancy         : 0.8134 ✓
  rouge_l                  : 0.1540 ✗
  bert_score               : 0.7729 ✓
  latency_sec              : 21.3332 ✓

DETAILED PER-QUERY TABLE
----------------------------------------------------------------------------------------------------------------------------------------------------
#    Query (truncated)                             Rec@K    Prec@K   CtxRel   Faith    AnsRel   ROUGE-L   BERTSc    Lat(s)
----------------------------------------------------------------------------------------------------------------------------------------------------
1    What is the maximum duration of a seafarer'.. 0.3333   0.2000   0.5654   0.5513   0.7925   0.1290    0.7623    35.05
2    What are the conditions for renewing or ext.. 0.0000   0.0000   

In [11]:
import gradio as gr

def chat_fn(user_message, history):
    if not user_message or not user_message.strip():
        history.append((user_message, "⚠️ Please enter a valid question."))
        return history, "No query provided."
    if len(user_message.strip()) < 5:
        history.append((user_message, "⚠️ Question too short."))
        return history, "Query too short."
    result  = generate_answer(user_message.strip())
    answer  = result["answer"]
    context = result["context"]
    latency = result["latency_sec"]
    history.append((user_message, f"{answer}\n\n⏱️ {latency}s"))
    return history, f"📄 **Retrieved Context (Top 5):**\n\n{context}" if context else "No context retrieved."

def clear_fn():
    return [], ""

with gr.Blocks(title=f"Semanual — {MODEL_DISPLAY_NAME}", theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"""
    # Seamanual — {MODEL_DISPLAY_NAME}
    ### RAG + CrossEncoder Reranker | POEA Maritime Labor Law
    Ask any question about the **POEA Standard Employment Contract for Filipino Seafarers**.
    """)
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Chat", height=500)
            with gr.Row():
                user_input = gr.Textbox(
                    placeholder="e.g. What are a seafarer's medical benefits if injured?",
                    label="Question", lines=2, scale=5
                )
                with gr.Column(scale=1):
                    submit_btn = gr.Button("Send ▶", variant="primary")
                    clear_btn  = gr.Button("Clear")
        with gr.Column(scale=2):
            context_box = gr.Markdown(value="*Retrieved context will appear here.*")
    gr.Markdown("---\n**Example:** *What is the maximum contract duration?* | *Can a seafarer claim overtime?* | *What happens on death during contract?*")
    submit_btn.click(fn=chat_fn, inputs=[user_input, chatbot],
                     outputs=[chatbot, context_box]).then(lambda: "", outputs=user_input)
    user_input.submit(fn=chat_fn, inputs=[user_input, chatbot],
                      outputs=[chatbot, context_box]).then(lambda: "", outputs=user_input)
    clear_btn.click(fn=clear_fn, outputs=[chatbot, context_box])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://feab0bdf8322712964.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
while True:
    question = input("You: ").strip()
    if question.lower() == "exit":
        print("Exiting.")
        break
    if not question:
        continue
    result  = generate_answer(question)
    answer  = result["answer"]
    chunks  = result["chunks"]
    latency = result["latency_sec"]
    print(f"\nBot: {answer}")
    print(f"\n CtxRel: {compute_context_relevancy(question, chunks):.4f} | "
          f"Faith: {compute_faithfulness(answer, chunks):.4f} | "
          f"AnsRel: {compute_answer_relevancy(question, answer):.4f} | "
          f"Lat: {latency}s\n")

You: what are the medical conditions of a seafarer

Bot: According to the given sources, the medical conditions listed include wearing life jackets, having a safety harness with attachment, and ensuring availability of a safety helmet during employment. Additionally, if required, obtaining permission before going outside the lifeline while the ship is moving. This highlights the importance of personal protective equipment and adherence to safety protocols.

 CtxRel: 0.6083 | Faith: 0.6734 | AnsRel: 0.7687 | Lat: 25.35s

